# OZZ — HALctf Autonomous Pentesting Agent (Kaggle Pipeline)
Pipeline autônomo com servidor Qwen 2.5 3B local na porta 8000 e Sandbox CTF Node.js na porta 3000.

In [ ]:
# Instalar dependências necessárias
!pip install -q fastapi uvicorn pydantic requests transformers torch accelerate
import os, sys, time, requests, subprocess, json, traceback
os.makedirs('/kaggle/working/hf_cache', exist_ok=True)
print('✅ Ambiente e diretórios configurados com sucesso!')

In [ ]:
# Clonar repositório Tretabolt/ozz-halctf e instalar dependências do agente
!if [ -d /kaggle/working/ozz-halctf ]; then rm -rf /kaggle/working/ozz-halctf; fi
!git clone https://github.com/Tretabolt/ozz-halctf.git /kaggle/working/ozz-halctf
os.chdir('/kaggle/working/ozz-halctf')
!pip install -q -r requirements.txt

# Subir Sandbox CTF Node.js na porta 3000 em background com resiliência
!if [ ! -d /kaggle/working/ctf-sandbox ]; then git clone https://github.com/kimdane/ctf.git /kaggle/working/ctf-sandbox || true; fi
!cd /kaggle/working/ctf-sandbox && (npm install --production --silent || true)
ctf_log = open('/kaggle/working/ctf.log', 'w', encoding='utf-8')
ctf_proc = subprocess.Popen(
    ['npm', 'start'],
    cwd='/kaggle/working/ctf-sandbox',
    env={**os.environ, 'PORT': '3000'},
    stdout=ctf_log,
    stderr=ctf_log
)
print('🚀 Processo da Sandbox CTF disparado em background na porta 3000!')

In [ ]:
# Criar script hf_server.py com cache em disco /kaggle/working/hf_cache e logs sem buffer
server_script = '''
import os, sys, torch, traceback
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List, Dict, Any, Optional, Union
from transformers import AutoModelForCausalLM, AutoTokenizer
import uvicorn

app = FastAPI()
model_id = "Qwen/Qwen2.5-Coder-3B-Instruct"
cache_dir = "/kaggle/working/hf_cache"
print("📥 Carregando modelo Qwen 2.5 3B no cache de disco...", flush=True)

try:
    tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir, trust_remote_code=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
    
    current_device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if current_device == "cuda" else torch.float32
    print(f"🎯 Dispositivo selecionado: {current_device} (dtype={dtype})", flush=True)
    
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        cache_dir=cache_dir,
        torch_dtype=dtype,
        device_map=current_device,
        trust_remote_code=True
    )
    print(f"✅ Modelo carregado com sucesso no dispositivo: {current_device}!", flush=True)
except Exception as e:
    print(f"❌ FALHA CRÍTICA AO CARREGAR MODELO: {e}", flush=True)
    traceback.print_exc()
    sys.exit(1)

class ChatRequest(BaseModel):
    model: str
    messages: List[Dict[str, str]]
    max_tokens: Optional[int] = 512
    temperature: Optional[float] = 0.3
    stop: Optional[Union[str, List[str]]] = None
    top_p: Optional[float] = None

@app.get("/v1/models")
def get_models():
    return {"data": [{"id": model_id}, {"id": "qwen2.5-coder-3b"}, {"id": "Qwen/Qwen2.5-Coder-7B-Instruct"}]}

@app.post("/v1/chat/completions")
def chat_completion(req: ChatRequest):
    global model, tokenizer, current_device
    try:
        try:
            prompt = tokenizer.apply_chat_template(req.messages, tokenize=False, add_generation_prompt=True)
        except Exception as t_err:
            print(f"⚠️ Fallback no chat template: {t_err}", flush=True)
            prompt = "\n".join([f"{m.get('role', 'user')}: {m.get('content', '')}" for m in req.messages])
        
        max_tok = min(req.max_tokens or 512, 512)
        print(f"🔍 [LLM Req] prompt length={len(prompt)}, max_tokens={max_tok}, device={current_device}", flush=True)
        
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(current_device)
        temp = req.temperature if req.temperature is not None else 0.3
        gen_kwargs = {
            "max_new_tokens": max_tok,
            "do_sample": True if temp > 0 else False,
            "pad_token_id": tokenizer.pad_token_id or tokenizer.eos_token_id
        }
        if temp > 0:
            gen_kwargs["temperature"] = temp
        
        with torch.no_grad():
            outputs = model.generate(**inputs, **gen_kwargs)
        generated_ids = outputs[0][inputs.input_ids.shape[1]:]
        response_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
        print(f"✅ [LLM Resp] gerou {len(response_text)} caracteres!", flush=True)
        return {"choices": [{"message": {"role": "assistant", "content": response_text}}]}
    except Exception as err:
        print(f"❌ Erro interno ao gerar resposta da LLM: {err}", flush=True)
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=f"Internal generation error: {str(err)}")

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

with open("/kaggle/working/hf_server.py", "w", encoding="utf-8") as f:
    f.write(server_script)

# Iniciar servidor FastAPI em background
log_file = open("/kaggle/working/hf_server.log", "w", encoding="utf-8")
server_proc = subprocess.Popen([
    "python3", "-u", "/kaggle/working/hf_server.py"
], stdout=log_file, stderr=log_file, env={**os.environ, "PYTHONUNBUFFERED": "1"})
print("⚡ Servidor LLM FastAPI disparado na porta 8000 (cache: /kaggle/working/hf_cache)!")

In [ ]:
import os, sys, time, requests, subprocess, json

# Health check: Aguardar servidor Qwen (porta 8000)
print('⏳ Aguardando servidor Qwen (http://localhost:8000/v1/models)...')
server_ready = False
for i in range(30):
    try:
        r = requests.get('http://localhost:8000/v1/models', timeout=2)
        if r.status_code == 200:
            print('✅ Servidor Qwen pronto!')
            server_ready = True
            break
    except Exception:
        pass
    time.sleep(5)

# Se o servidor não iniciou, imprimir o log IMEDIATAMENTE e interromper
if not server_ready:
    print('\n❌ SERVIDOR QWEN NÃO INICIOU NA PORTA 8000!')
    if os.path.exists('/kaggle/working/hf_server.log'):
        with open('/kaggle/working/hf_server.log', 'r', encoding='utf-8', errors='ignore') as f:
            print('======== LOGS DE INICIALIZAÇÃO DO SERVIDOR QWEN ========')
            print(f.read())
            print('========================================================')
    raise RuntimeError('Servidor Qwen falhou ao iniciar na porta 8000')

# Pre-flight Smoke Test do Endpoint /v1/chat/completions
print('🧪 Testando endpoint /v1/chat/completions antes de iniciar o agente...')
try:
    test_resp = requests.post(
        'http://localhost:8000/v1/chat/completions',
        json={
            'model': 'Qwen/Qwen2.5-Coder-3B-Instruct',
            'messages': [{'role': 'user', 'content': 'Diga oi'}],
            'max_tokens': 50,
            'temperature': 0.1
        },
        timeout=60
    )
    print(f'STATUS TESTE LLM: {test_resp.status_code}')
    print(f'RESPOSTA TESTE LLM: {test_resp.text}')
except Exception as e:
    print(f'❌ Teste inicial de LLM falhou: {e}')

# Health check: Aguardar Sandbox CTF (porta 3000)
print('⏳ Aguardando Sandbox CTF (http://localhost:3000)...')
for i in range(30):
    try:
        r = requests.get('http://localhost:3000', timeout=2)
        if r.status_code in [200, 301, 302, 403, 404]:
            print('✅ Sandbox CTF pronta!')
            break
    except Exception:
        pass
    time.sleep(2)

# Executar Agente autônomo Ozz MNHI 3.5 com limite sustentável de iterações
print('🚀 Disparando o agente Ozz MNHI 3.5...')
env_vars = {**os.environ, 'MAX_ITERATIONS': '50'}
agent_result = subprocess.run(['python3', '-m', 'agent', 'http://localhost:3000'], env=env_vars, check=False)
print(f'🏁 Agente finalizou com código de saída: {agent_result.returncode}')

In [ ]:
# Relatório final e leitura explícita do log do servidor Qwen
import os, sys, json

try:
    if 'log_file' in globals() and not log_file.closed:
        log_file.flush()
    with open('/kaggle/working/hf_server.log', 'r', encoding='utf-8', errors='ignore') as f:
        logs = f.read()
        print('\n======== LOGS INTERNOS DO SERVIDOR QWEN (hf_server.log) ========')
        print(logs[-4000:] if len(logs) > 4000 else logs)
        print('==================================================================\n')
except Exception as e:
    print(f'⚠️ Erro ao ler log do servidor: {e}')

sandbox_running = 'ctf_proc' in globals() and ctf_proc.poll() is None
agent_exit_code = agent_result.returncode if 'agent_result' in globals() else None
report = {
    "status": "SUCCESS" if agent_exit_code == 0 else "FINISHED",
    "agent": "Ozz MNHI 3.5",
    "model": "Qwen2.5-Coder-3B-Instruct",
    "sandbox_running": sandbox_running,
    "agent_exit_code": agent_exit_code,
}
with open("/kaggle/working/report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)
print("✅ Relatório salvo em /kaggle/working/report.json!")
if 'server_proc' in globals():
    server_proc.terminate()
if 'ctf_proc' in globals() and ctf_proc.poll() is None:
    ctf_proc.terminate()
if 'ctf_log' in globals():
    ctf_log.close()
